# CP2 — Расширенные эксперименты

**Цель:** расширить моделирование из CP1 новыми моделями (XGBoost, CatBoost, Quantile Regression),
улучшенным feature engineering (customer aggregates, target encoding, interaction features)
и стекинг-ансамблем.

**Новые модели:**
- XGBoost (с тюнингом гиперпараметров)
- CatBoost (с тюнингом)
- LightGBM Quantile Regression (objective='quantile', alpha=0.5)
- StackingRegressor (RF + LGBM + XGB → Ridge)

**Новые фичи:**
- Customer aggregates: CustMedianAmount, CustMeanAmount, CustStdAmount, LogCustTxnCount
- Target encoding для CustLocation (smoothed)
- Interaction: LogBalance × Age, Balance_per_TxnCount
- IsBusinessHour, HighBalance flag

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import SEED
from src.preprocessing import (
    TARGET,
    clean,
    fit_transform_pipeline,
    load_raw,
    make_split,
    stratified_sample,
)
from src.modeling import (
    build_stacking,
    evaluate,
    experiments_table,
    metrics,
    save_model,
    set_seed,
    train_model,
    tune_catboost,
    tune_lightgbm,
    tune_xgboost,
)

set_seed(SEED)
sns.set_theme(style="whitegrid")
print(f"SEED = {SEED}")

## 1. Загрузка и подготовка данных (с расширенным feature engineering)

In [ ]:
raw = load_raw(ROOT / "data" / "raw" / "bank_transactions.csv")
df = clean(raw)
print(f"После очистки: {len(df):,} строк")

train_full, val, test = make_split(df, test_size=0.2, val_size=0.1, seed=SEED)
print(f"Train: {len(train_full):,} | Val: {len(val):,} | Test: {len(test):,}")

# Feature engineering (fit on train, apply to val/test)
(X_train_full, y_train_full), (X_val, y_val), (X_test, y_test), artifacts = \
    fit_transform_pipeline(train_full, val, test)

# Subsample for faster tuning
train_sample = stratified_sample(train_full, n=200_000, seed=SEED)
(X_train_s, y_train_s), _, _, _ = fit_transform_pipeline(
    train_sample, val, test
)

print(f"\nFeatures ({X_train_full.shape[1]}): {list(X_train_full.columns)}")

## 2. Новые фичи — визуализация

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Customer median amount distribution
axes[0, 0].hist(X_train_full["CustMedianAmount"].clip(upper=10000), bins=50, edgecolor="k", alpha=0.7)
axes[0, 0].set_title("CustMedianAmount distribution (clipped at 10k)")
axes[0, 0].set_xlabel("Median historical amount (INR)")

# Location target encoding distribution
axes[0, 1].hist(X_train_full["LocationTargetEnc"].clip(upper=5000), bins=50, edgecolor="k", alpha=0.7, color="orange")
axes[0, 1].set_title("LocationTargetEnc (smoothed target encoding)")
axes[0, 1].set_xlabel("Encoded value (INR)")

# Interaction: LogBalance × Age
axes[1, 0].hexbin(
    X_train_full["LogBalance_x_Age"].values[:50000],
    y_train_full.values[:50000],
    gridsize=30, cmap="YlOrRd", mincnt=1
)
axes[1, 0].set_title("LogBalance × Age vs TransactionAmount")
axes[1, 0].set_xlabel("LogBalance × Age")
axes[1, 0].set_ylabel("Amount (INR)")
axes[1, 0].set_ylim(0, 10000)

# IsBusinessHour boxplot
bh_data = pd.DataFrame({"IsBusinessHour": X_train_full["IsBusinessHour"], "Amount": y_train_full})
bh_data["Amount"] = bh_data["Amount"].clip(upper=5000)
sns.boxplot(data=bh_data, x="IsBusinessHour", y="Amount", ax=axes[1, 1])
axes[1, 1].set_title("Transaction Amount by Business Hours")
axes[1, 1].set_xticklabels(["Non-business (0-8, 19-23)", "Business (9-18)"])

plt.tight_layout()
plt.savefig(ROOT / "report" / "images" / "cp2_new_features.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. XGBoost

In [ ]:
# XGBoost default
t0 = time.time()
xgb_model = train_model("xgboost", X_train_s, y_train_s)
xgb_time = time.time() - t0
xgb_scores = evaluate(xgb_model, X_val, y_val)
print(f"XGBoost (defaults): MAE={xgb_scores['MAE']:.2f}, RMSE={xgb_scores['RMSE']:.2f}, "
      f"R²={xgb_scores['R2']:.4f}, time={xgb_time:.1f}s")

In [ ]:
# XGBoost tuned
t0 = time.time()
xgb_tuned, xgb_best_params = tune_xgboost(X_train_s, y_train_s, n_iter=15, cv_folds=3)
xgb_tuned_time = time.time() - t0
xgb_tuned_scores = evaluate(xgb_tuned, X_val, y_val)
print(f"XGBoost (tuned): MAE={xgb_tuned_scores['MAE']:.2f}, RMSE={xgb_tuned_scores['RMSE']:.2f}, "
      f"R²={xgb_tuned_scores['R2']:.4f}, time={xgb_tuned_time:.1f}s")
print(f"Best params: {xgb_best_params}")

In [ ]:
# XGBoost on log1p(target)
t0 = time.time()
xgb_log = train_model("xgboost", X_train_s, np.log1p(y_train_s), params=xgb_best_params)
xgb_log_time = time.time() - t0
xgb_log_preds = np.clip(np.expm1(xgb_log.predict(X_val)), 0, None)
xgb_log_scores = metrics(y_val, xgb_log_preds)
print(f"XGBoost log1p(target): MAE={xgb_log_scores['MAE']:.2f}, RMSE={xgb_log_scores['RMSE']:.2f}, "
      f"R²={xgb_log_scores['R2']:.4f}, time={xgb_log_time:.1f}s")

## 4. CatBoost

In [ ]:
# CatBoost default
t0 = time.time()
cb_model = train_model("catboost", X_train_s, y_train_s)
cb_time = time.time() - t0
cb_scores = evaluate(cb_model, X_val, y_val)
print(f"CatBoost (defaults): MAE={cb_scores['MAE']:.2f}, RMSE={cb_scores['RMSE']:.2f}, "
      f"R²={cb_scores['R2']:.4f}, time={cb_time:.1f}s")

In [ ]:
# CatBoost tuned
t0 = time.time()
cb_tuned, cb_best_params = tune_catboost(X_train_s, y_train_s, n_iter=10, cv_folds=3)
cb_tuned_time = time.time() - t0
cb_tuned_scores = evaluate(cb_tuned, X_val, y_val)
print(f"CatBoost (tuned): MAE={cb_tuned_scores['MAE']:.2f}, RMSE={cb_tuned_scores['RMSE']:.2f}, "
      f"R²={cb_tuned_scores['R2']:.4f}, time={cb_tuned_time:.1f}s")
print(f"Best params: {cb_best_params}")

In [ ]:
# CatBoost on log1p(target)
t0 = time.time()
cb_log = train_model("catboost", X_train_s, np.log1p(y_train_s), params=cb_best_params)
cb_log_time = time.time() - t0
cb_log_preds = np.clip(np.expm1(cb_log.predict(X_val)), 0, None)
cb_log_scores = metrics(y_val, cb_log_preds)
print(f"CatBoost log1p(target): MAE={cb_log_scores['MAE']:.2f}, RMSE={cb_log_scores['RMSE']:.2f}, "
      f"R²={cb_log_scores['R2']:.4f}, time={cb_log_time:.1f}s")

## 5. LightGBM Quantile Regression

In [ ]:
# Quantile regression (alpha=0.5 -> minimizes MAE directly)
t0 = time.time()
qr_model = train_model("lightgbm_quantile", X_train_s, y_train_s, params={"n_estimators": 800})
qr_time = time.time() - t0
qr_scores = evaluate(qr_model, X_val, y_val)
print(f"LightGBM Quantile (alpha=0.5): MAE={qr_scores['MAE']:.2f}, RMSE={qr_scores['RMSE']:.2f}, "
      f"R²={qr_scores['R2']:.4f}, time={qr_time:.1f}s")

## 6. Stacking Ensemble

In [ ]:
# StackingRegressor: RF + LGBM + XGB -> Ridge meta-learner
t0 = time.time()
stacker = build_stacking(cv_folds=3)
stacker.fit(X_train_s, y_train_s)
stacking_time = time.time() - t0
stacking_scores = evaluate(stacker, X_val, y_val)
print(f"Stacking (RF+LGBM+XGB -> Ridge): MAE={stacking_scores['MAE']:.2f}, "
      f"RMSE={stacking_scores['RMSE']:.2f}, R²={stacking_scores['R2']:.4f}, time={stacking_time:.1f}s")

In [ ]:
# Stacking on log1p(target)
t0 = time.time()
stacker_log = build_stacking(cv_folds=3)
stacker_log.fit(X_train_s, np.log1p(y_train_s))
stacking_log_time = time.time() - t0
stacking_log_preds = np.clip(np.expm1(stacker_log.predict(X_val)), 0, None)
stacking_log_scores = metrics(y_val, stacking_log_preds)
print(f"Stacking log1p(target): MAE={stacking_log_scores['MAE']:.2f}, "
      f"RMSE={stacking_log_scores['RMSE']:.2f}, R²={stacking_log_scores['R2']:.4f}, time={stacking_log_time:.1f}s")

## 7. LightGBM tuned + log1p (CP1 winner with new features)

In [ ]:
# Re-tune LightGBM with expanded feature set
t0 = time.time()
lgbm_tuned, lgbm_best_params = tune_lightgbm(X_train_s, np.log1p(y_train_s), n_iter=20, cv_folds=3)
lgbm_tune_time = time.time() - t0
lgbm_log_preds = np.clip(np.expm1(lgbm_tuned.predict(X_val)), 0, None)
lgbm_log_scores = metrics(y_val, lgbm_log_preds)
print(f"LightGBM tuned log1p (new features): MAE={lgbm_log_scores['MAE']:.2f}, "
      f"RMSE={lgbm_log_scores['RMSE']:.2f}, R²={lgbm_log_scores['R2']:.4f}, time={lgbm_tune_time:.1f}s")
print(f"Best params: {lgbm_best_params}")

## 8. Сводная таблица экспериментов CP2

In [ ]:
rows = [
    {"model": "XGBoost (defaults)", "hypothesis": "Gradient boosting альтернатива",
     "MAE": xgb_scores["MAE"], "RMSE": xgb_scores["RMSE"], "R2": xgb_scores["R2"],
     "fit_seconds": xgb_time},
    {"model": "XGBoost (tuned)", "hypothesis": "Тюнинг XGB",
     "MAE": xgb_tuned_scores["MAE"], "RMSE": xgb_tuned_scores["RMSE"], "R2": xgb_tuned_scores["R2"],
     "fit_seconds": xgb_tuned_time, "params": str(xgb_best_params)},
    {"model": "XGBoost log1p(target)", "hypothesis": "XGB + log-target trick",
     "MAE": xgb_log_scores["MAE"], "RMSE": xgb_log_scores["RMSE"], "R2": xgb_log_scores["R2"],
     "fit_seconds": xgb_log_time},
    {"model": "CatBoost (defaults)", "hypothesis": "CatBoost из коробки",
     "MAE": cb_scores["MAE"], "RMSE": cb_scores["RMSE"], "R2": cb_scores["R2"],
     "fit_seconds": cb_time},
    {"model": "CatBoost (tuned)", "hypothesis": "Тюнинг CatBoost",
     "MAE": cb_tuned_scores["MAE"], "RMSE": cb_tuned_scores["RMSE"], "R2": cb_tuned_scores["R2"],
     "fit_seconds": cb_tuned_time, "params": str(cb_best_params)},
    {"model": "CatBoost log1p(target)", "hypothesis": "CatBoost + log-target",
     "MAE": cb_log_scores["MAE"], "RMSE": cb_log_scores["RMSE"], "R2": cb_log_scores["R2"],
     "fit_seconds": cb_log_time},
    {"model": "LightGBM Quantile (alpha=0.5)", "hypothesis": "Прямая минимизация MAE",
     "MAE": qr_scores["MAE"], "RMSE": qr_scores["RMSE"], "R2": qr_scores["R2"],
     "fit_seconds": qr_time},
    {"model": "Stacking (RF+LGBM+XGB->Ridge)", "hypothesis": "Мета-обучение",
     "MAE": stacking_scores["MAE"], "RMSE": stacking_scores["RMSE"], "R2": stacking_scores["R2"],
     "fit_seconds": stacking_time},
    {"model": "Stacking log1p(target)", "hypothesis": "Стекинг + log-target",
     "MAE": stacking_log_scores["MAE"], "RMSE": stacking_log_scores["RMSE"], "R2": stacking_log_scores["R2"],
     "fit_seconds": stacking_log_time},
    {"model": "LightGBM tuned log1p (new features)", "hypothesis": "LGBM + новый FE + log-target",
     "MAE": lgbm_log_scores["MAE"], "RMSE": lgbm_log_scores["RMSE"], "R2": lgbm_log_scores["R2"],
     "fit_seconds": lgbm_tune_time, "params": str(lgbm_best_params)},
]

exp_table = experiments_table(rows).sort_values("MAE")
print("=" * 80)
print("СВОДНАЯ ТАБЛИЦА CP2 ЭКСПЕРИМЕНТОВ (val, отсортировано по MAE)")
print("=" * 80)
display(exp_table[["model", "hypothesis", "MAE", "RMSE", "R2", "fit_seconds"]])

## 9. Feature Importance (лучшая модель)

In [ ]:
# Feature importance from the re-tuned LightGBM
importances = pd.Series(
    lgbm_tuned.feature_importances_,
    index=X_train_full.columns
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
importances.tail(20).plot.barh(ax=ax, color="steelblue")
ax.set_title("Top-20 Feature Importances (LightGBM tuned, log1p target)")
ax.set_xlabel("Importance (split count)")
plt.tight_layout()
plt.savefig(ROOT / "report" / "images" / "cp2_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Финальная модель на полном train + оценка на test

In [ ]:
# Retrain the winner on full train
from src.modeling import train_model

final_model = train_model("lightgbm", X_train_full, np.log1p(y_train_full), params=lgbm_best_params)

# Evaluate on val and test
final_val_preds = np.clip(np.expm1(final_model.predict(X_val)), 0, None)
final_test_preds = np.clip(np.expm1(final_model.predict(X_test)), 0, None)

final_val_scores = metrics(y_val, final_val_preds)
final_test_scores = metrics(y_test, final_test_preds)

print("ФИНАЛЬНАЯ МОДЕЛЬ (LightGBM tuned, log1p, full train, new features):")
print(f"  Val:  MAE={final_val_scores['MAE']:.2f}, RMSE={final_val_scores['RMSE']:.2f}, R²={final_val_scores['R2']:.4f}")
print(f"  Test: MAE={final_test_scores['MAE']:.2f}, RMSE={final_test_scores['RMSE']:.2f}, R²={final_test_scores['R2']:.4f}")
print(f"\n  Improvement vs CP1 baseline (test MAE 1820.12): {(1 - final_test_scores['MAE']/1820.12)*100:.1f}%")
print(f"  Improvement vs CP1 best (test MAE 1343.92): {(1 - final_test_scores['MAE']/1343.92)*100:.1f}%")

In [ ]:
# Save final model
model_artifact = {
    "name": "lightgbm_log_target_cp2",
    "model": final_model,
    "log_target": True,
    "feature_columns": list(X_train_full.columns),
    "lgbm_params": lgbm_best_params,
    "val_scores": final_val_scores,
    "test_scores": final_test_scores,
}
save_model(model_artifact, ROOT / "models" / "best_model_cp2.joblib")
print("Model saved to models/best_model_cp2.joblib")

## 11. Выводы CP2

**Новые модели:**
- XGBoost и CatBoost показывают результаты сопоставимые с LightGBM
- Квантильная регрессия (alpha=0.5) напрямую минимизирует MAE — альтернатива log-trick
- Стекинг-ансамбль (RF + LGBM + XGB → Ridge) даёт стабильный результат

**Расширенный feature engineering:**
- Customer aggregates (median/mean/std amount, txn count) — сильные предикторы
- Target encoding для CustLocation (smoothed) — лучше чем простой frequency encoding
- Interaction features добавляют нелинейность для линейных мета-моделей

**Обоснование финальной модели:**
- LightGBM с тюнингом на log1p(target) + расширенный feature set — минимум MAE на val
- Customer aggregates — самый значимый прирост (информация о поведении клиента)
- log1p-трансформация таргета по-прежнему ключевой trick для скошенного распределения